Satellite Chlorophyll-a Gap Filling using Temporal Interpolation

In [1]:
#import necessary libraries
import ee
import geemap
import numpy as np
import matplotlib.pyplot as plt

In [2]:
#Authenticate earth engine
ee.Authenticate()

In [3]:
# 1. Initialize Earth Engine
ee.Initialize(project='ee-punyaravichandran')

In [4]:
# 2. Create Map
Map = geemap.Map(center=(10, 80), zoom=5)
Map.basemap_demo()
Map

Map(center=[10, 80], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(chil…

In [5]:
# 3. Load Region of Interest (Shapefile from EE Assets)
roi_fc = ee.FeatureCollection("projects/ee-punyaravichandran/assets/ion")
roi = roi_fc.geometry()

In [6]:
# 4. Load MODIS-Aqua Chlorophyll-a Dataset
dataset = (
    ee.ImageCollection('NASA/OCEANDATA/MODIS-Aqua/L3SMI')
    .select('chlor_a')
    .filterDate('2002-08-01', '2021-06-30')
    .filterBounds(roi)
)

In [7]:
# Clip images to ROI
dataset = dataset.map(lambda img: img.clip(roi))

In [8]:
# 5. Mask Handling Function
def mask_chlorophyll(image):
    masked = image.updateMask(image.mask())
    return masked.copyProperties(image, ['system:time_start'])

dataset = dataset.map(mask_chlorophyll)

In [9]:
# 6. Add Timestamp Band
def add_timestamp(image):
    timestamp = image.metadata('system:time_start').rename('timestamp')
    timestamp = timestamp.updateMask(image.mask())
    return image.addBands(timestamp)

dataset = dataset.map(add_timestamp)

In [10]:
# 7. Temporal Join Configuration
days = 200
millis = ee.Number(days).multiply(1000 * 60 * 60 * 24)

max_diff = ee.Filter.maxDifference(
    difference=millis,
    leftField='system:time_start',
    rightField='system:time_start'
)

before_filter = ee.Filter.lessThanOrEquals(
    leftField='system:time_start',
    rightField='system:time_start'
)

after_filter = ee.Filter.greaterThanOrEquals(
    leftField='system:time_start',
    rightField='system:time_start'
)

In [11]:
# Join images AFTER
join_after = ee.Join.saveAll(
    matchesKey='after',
    ordering='system:time_start',
    ascending=False
)

after_joined = join_after.apply(
    primary=dataset,
    secondary=dataset,
    condition=ee.Filter.And(max_diff, before_filter)
)

In [12]:
# Join images BEFORE
join_before = ee.Join.saveAll(
    matchesKey='before',
    ordering='system:time_start',
    ascending=True
)

joined = join_before.apply(
    primary=after_joined,
    secondary=after_joined,
    condition=ee.Filter.And(max_diff, after_filter)
)

In [13]:
# 8. Temporal Interpolation Function
def interpolate_image(image):
    image = ee.Image(image)

    before_imgs = ee.ImageCollection.fromImages(
        ee.List(image.get('before'))
    ).mosaic()

    after_imgs = ee.ImageCollection.fromImages(
        ee.List(image.get('after'))
    ).mosaic()

    # Convert timestamps to proper images
    t1 = before_imgs.select('timestamp').rename('t1')
    t2 = after_imgs.select('timestamp').rename('t2')
    t  = ee.Image(image.get('system:time_start')).rename('t')

    # Stack time bands
    time_stack = ee.Image.cat([t1, t2, t])

    ratio = time_stack.expression(
        '(t - t1) / (t2 - t1)', {
            't': time_stack.select('t'),
            't1': time_stack.select('t1'),
            't2': time_stack.select('t2')
        }
    )
    ratio = ratio.where(t2.subtract(t1).eq(0), 0)

    interpolated = before_imgs.add(
        after_imgs.subtract(before_imgs).multiply(ratio)
    )

    return image.unmask(interpolated).copyProperties(
        image, ['system:time_start']
    )


In [14]:
# Apply interpolation
gap_filled = ee.ImageCollection(joined.map(interpolate_image))

In [19]:
#check the number of Images
print(gap_filled.size().getInfo())

793


In [20]:
# 9. Visualization
vis_params = {
    'min': 0.05,
    'max': 20,
    'palette': ['blue', 'cyan', 'green', 'yellow', 'red'],
    'scale': 4000
}

Map.addLayer(gap_filled.mean(), vis_params, 'Gap-filled Chlorophyll-a', True, 0.7)
Map